In [1]:
import pandas as pd
sintomas_doencas = pd.read_excel("https://raw.githubusercontent.com/ppsj1983/Etapa_3_Ex_2/main/doencas_cardiacas.xlsx")

In [2]:
import pandas as pd
relato_paciente = pd.read_csv("https://raw.githubusercontent.com/ppsj1983/Etapa_3_Ex_2/main/guia_saude_cardiaca_1.txt")

In [3]:
import pandas as pd
ref_avaliacao = pd.read_excel("https://raw.githubusercontent.com/ppsj1983/Etapa_3_Ex_2/main/ref_avaliacao.xlsx")

In [4]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

STOP_WORDS_PT = [
    'tudo', 'comecou', 'cerca', 'de', 'tres', 'semanas', 'depois', 'ter', 'feito', 'um',
    'sem', 'tomar', 'apesar', 'eu', 'ja', 'com', 'que', 'nao', 'passava', 'acompanhada',
    'no', 'na', 'nas', 'me', 'sentia', 'extremamente', 'notei', 'o', 'aparecimento', 'pequenas']

def limpar_texto(texto):
    """Remove caracteres especiais, pontuação e padroniza o texto."""
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', ' ', texto) # Remove pontuação
    return texto
lista_sintomas = list(set(sintomas_doencas["Sintoma_1"].tolist() + sintomas_doencas["Sintoma_2"].tolist()))
lista_sintomas_limpos = [limpar_texto(s) for s in lista_sintomas]
vectorizer = TfidfVectorizer(
    analyzer="word", 
    ngram_range=(1, 2), # Reduzido para 2 para focar em expressões-chave mais curtas e precisas
    stop_words=STOP_WORDS_PT)  
tfidf_sintomas = vectorizer.fit_transform(lista_sintomas_limpos)
def identificar_sintoma_e_doenca_melhorado(relato):
    relato_limpo = limpar_texto(relato)
    tfidf_relato = vectorizer.transform([relato_limpo])
    similaridades = cosine_similarity(tfidf_relato, tfidf_sintomas).flatten()
    indice_melhor_match = np.argmax(similaridades)
    maior_score = similaridades[indice_melhor_match]
    # Regulador de score de captura"
    if maior_score < 0.25:
        return "Sintoma Complexo / Inconclusivo", "Encaminhar para Avaliação Médica Detalhada"       
    sintoma_identificado = lista_sintomas[indice_melhor_match]
    filtro_doenca = sintomas_doencas[
        (sintomas_doencas["Sintoma_1"] == sintoma_identificado)
        | (sintomas_doencas["Sintoma_2"] == sintoma_identificado)
    ]
    
    if not filtro_doenca.empty:
        doenca = filtro_doenca["Doenca_Associada"].values[0]
    else:
        doenca = "Desconhecida"
        
    return sintoma_identificado, doenca

resultados = relato_paciente["Relato"].apply(identificar_sintoma_e_doenca_melhorado)
relato_paciente["Sintoma_Principal"] = [r[0] for r in resultados]
relato_paciente["Doenca_Associada"] = [r[1] for r in resultados]



In [5]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

STOP_WORDS_PT = [
    "a", "ao", "aos", "aquela", "aquelas", "aquele", "aqueles", "aquilo", "as", "até", "com", 
    "como", "da", "das", "de", "dela", "delas", "dele", "deles", "depois", "do", "dos", "e", "é", 
    "em", "entre", "era", "eram", "essa", "essas", "esse", "esses", "esta", "estamos", "estão",
      "estas", "este", "estes", "estou", "eu", "foi", "fomos", "for", "foram", "fosse", "fossem", 
      "fui", "há", "haja", "hajam", "houve", "houvemos", "houveram", "ia", "iam", "isso", "isto",
        "já", "lhe", "lhes", "mais", "mas", "me", "mesmo", "meu", "meus", "minha", "minhas", 
        "muito", "na", "nas", "nem", "no", "nos", "nossa", "nossas", "nosso", "nossos", "num", 
        "numa", "o", "os", "ou", "para", "pela", "pelas", "pelo", "pelos", "por", "qual", 
        "quando", "que", "quem", "se", "seja", "sejam", "sem", "ser", "será", "serão", 
        "serei", "seria", "seriam", "seu", "seus", "só", "sob", "sobre", "sua", "suas", "também", 
        "te", "tem", "tém", "temos", "tenha", "tenham", "tenho", "terá", "terão", "terei", 
        "teria", "teriam", "teu", "teus", "teve", "tinha", "tinham", "tive", "tivemos", 
        "tiveram", "tivesse", "tivessem", "tu", "tua", "tuas", "um", "uma", "umas", "uns", "você", "vocês", "vos"
]

def classificar_e_rastrear_origem(relato, df_ref):
    relato_limpo = re.sub(r'[.,\/#!$%\^&\*;:{}=\-_`~()]', ' ', str(relato))
    relato_lower = relato_limpo.lower()
    correspondencias = []
    
    for _, row in df_ref.iterrows():
        palavra = str(row['palavra_referencia']).lower().strip()
        
        if palavra in relato_lower or any(palavra[:5] in token for token in relato_lower.split() if len(palavra) >= 5):
            correspondencias.append({
                'nivel': str(row['nivel_sintoma']).strip().capitalize(),
                'palavra': row['palavra_referencia']
            })      
            
    if not correspondencias:
        return "Não Classificado", "Nenhuma"    
        
    df_temp = pd.DataFrame(correspondencias)
    
    if df_temp['nivel'].str.lower().isin(['forte', 'grave']).any():
        escolha = df_temp[df_temp['nivel'].str.lower().isin(['forte', 'grave'])].iloc[0]
    elif df_temp['nivel'].str.lower().isin(['moderado']).any():
        escolha = df_temp[df_temp['nivel'].str.lower().isin(['moderado'])].iloc[0]
    else:
        escolha = df_temp.iloc[0]
        
    return escolha['nivel'], escolha['palavra']
resultados_rotulacao = relato_paciente["Relato"].apply(lambda x: classificar_e_rastrear_origem(x, ref_avaliacao))
relato_paciente["nivel_sintoma"] = [r[0] for r in resultados_rotulacao]
relato_paciente["Palavra_Chave_Referencia"] = [r[1] for r in resultados_rotulacao]
dados_treino = relato_paciente[relato_paciente["nivel_sintoma"] != "Não Classificado"].copy()
print("Distribuição inicial das classes obtidas via palavras-chave:")
print(dados_treino["nivel_sintoma"].value_counts())
contagem_classes = dados_treino["nivel_sintoma"].value_counts()
classes_validas = contagem_classes[contagem_classes >= 2].index
dados_treino = dados_treino[dados_treino["nivel_sintoma"].isin(classes_validas)].copy()
vectorizer_gravidade = TfidfVectorizer(analyzer="word", ngram_range=(1, 3), stop_words=STOP_WORDS_PT)
X = vectorizer_gravidade.fit_transform(dados_treino["Relato"])
y = dados_treino["nivel_sintoma"]
# Divisão Estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Configuração e Treinamento do Modelo
modelo_triagem = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    class_weight="balanced"
)
modelo_triagem.fit(X_train, y_train)

y_pred = modelo_triagem.predict(X_test)
print("\n--- Desempenho do Modelo nos Dados de Teste ---")
print(classification_report(y_test, y_pred))

dados_nao_classificados = relato_paciente[relato_paciente["nivel_sintoma"] == "Não Classificado"].copy()

if not dados_nao_classificados.empty:
    X_novos = vectorizer_gravidade.transform(dados_nao_classificados["Relato"])
    predicoes_novas = modelo_triagem.predict(X_novos)
    
    relato_paciente.loc[relato_paciente["nivel_sintoma"] == "Não Classificado", "Palavra_Chave_Referencia"] = "Predito pelo Modelo"
    relato_paciente.loc[relato_paciente["nivel_sintoma"] == "Não Classificado", "nivel_sintoma"] = predicoes_novas

print("\nProcessamento concluído com sucesso!")



Distribuição inicial das classes obtidas via palavras-chave:
Forte    5
Fraco    4
Name: nivel_sintoma, dtype: int64

--- Desempenho do Modelo nos Dados de Teste ---
              precision    recall  f1-score   support

       Forte       0.00      0.00      0.00         1
       Fraco       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2


Processamento concluído com sucesso!


C:\Users\A0050637\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\A0050637\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\A0050637\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------------------------------------------------
# ETAPA ISOLADA: ANÁLISE DETALHADA DE PALAVRAS E MÉTRICAS DE ACURÁCIA
# -------------------------------------------------------------------------

def extrair_testes_individuais(relato, df_ref):
    """
    Varre o relato e mapeia todas as palavras de referência encontradas,
    retornando o detalhe estruturado de cada teste individual.
    """
    import re
    relato_limpo = re.sub(r'[.,\/#!$%\^&\*;:{}=\-_`~()]', ' ', str(relato))
    relato_lower = relato_limpo.lower()
    correspondencias = []
    
    for _, row in df_ref.iterrows():
        palavra = str(row['palavra_referencia']).lower().strip()
        nivel = str(row['nivel_sintoma']).strip().capitalize()
        
        # Validação por substring ou radical aproximado (ex: "pequena" encontra "pequenina")
        if palavra in relato_lower or any(palavra[:5] in token for token in relato_lower.split() if len(palavra) >= 5):
            correspondencias.append(f"[{row['palavra_referencia']} -> {nivel}]")
            
    if not correspondencias:
        return "Nenhuma correspondência encontrada"
        
    return "; ".join(correspondencias)

# 1. Adiciona a coluna com o resultado do teste individual para cada palavra localizada
relato_paciente["Detalhe_Palavras_Localizadas"] = relato_paciente["Relato"].apply(
    lambda x: extrair_testes_individuais(x, ref_avaliacao)
)

print(" -> Coluna 'Detalhe_Palavras_Localizadas' adicionada com sucesso ao DataFrame.")

# 2. Cálculo e exibição do nível de acurácia (utilizando as variáveis geradas no treino)
# Nota: Certifique-off que as variáveis y_test e y_pred do seu treino estão na memória.
try:
    acuracia_teste = accuracy_score(y_test, y_pred)
    
    print("\n" + "="*46)
    print(f" NÍVEL DE ACURÁCIA DO MODELO: {acuracia_teste * 100:.2f}%")
    print("="*46)
    print("\nRelatório de Classificação Detalhado:")
    print(classification_report(y_test, y_pred))

except NameError:
    print("\n[Aviso] Não foi possível calcular a acurácia automática.")
    print("Certifique-se de que o modelo foi executado e que as variáveis 'y_test' e 'y_pred' existem no ambiente.")


 -> Coluna 'Detalhe_Palavras_Localizadas' adicionada com sucesso ao DataFrame.

 NÍVEL DE ACURÁCIA DO MODELO: 50.00%

Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

       Forte       0.00      0.00      0.00         1
       Fraco       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



C:\Users\A0050637\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\A0050637\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\A0050637\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [8]:
import os

# Caminho especificado para o projeto da FIAP
diretorio_destino = r"C:\Users\A0050637\Documents\01_Diversos\FIAP\Etapa_3_Ex_2"

# Cria a estrutura de pastas caso ainda não exista no computador
os.makedirs(diretorio_destino, exist_ok=True)

# Define o caminho completo do ficheiro Excel
caminho_completo = os.path.join(diretorio_destino, "relatorio_final_relato_paciente.xlsx")

# Exporta o DataFrame para o formato Excel (.xlsx)
relato_paciente.to_excel(caminho_completo, index=False, engine="openpyxl")

print(f"Ficheiro Excel gerado com sucesso em:\n{caminho_completo}")




Ficheiro Excel gerado com sucesso em:
C:\Users\A0050637\Documents\01_Diversos\FIAP\Etapa_3_Ex_2\relatorio_final_relato_paciente.xlsx
